# NB07a: Create Difficulty-Stratified 80/20 Split

**Purpose:** Create a reproducible 80/20 train/val split stratified by combined difficulty (tumor volume + tumor/kidney ratio) for ALL 208 KiTS19 cases.

**Why tumor volume for stratification:**
- Tumor size is the #1 predictor of segmentation difficulty (KiTS19 challenge paper, p=4.8×10⁻¹¹)
- Available for ALL 208 cases (from GT labels) — unlike Dice scores which only exist for old 70-case val
- Small tumors = hard cases, large tumors = easy cases

**What this notebook produces:**
- `fold_splits_80_20.json` — the new split definition
- Analysis of val set composition and expected difficulty
- NO model training — just data analysis

**Previous split:** 138 train / 70 val (66/34)  
**New split:** ~168 train / ~42 val (80/20) — standard evaluation protocol in medical image segmentation

In [1]:
import numpy as np
import json
from pathlib import Path
from collections import Counter

# Paths
PREPROC_DIR = Path(r"D:\fsdf_uca_kits\outputs\preprocessed")
OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
STATS_DIR = OUTPUT_DIR / "stats"

# Create output directories for new split
(OUTPUT_DIR / "results_80_20").mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints_80_20").mkdir(exist_ok=True)

print(f"Preprocessed dir: {PREPROC_DIR}")
print(f"Stats dir: {STATS_DIR}")

Preprocessed dir: D:\fsdf_uca_kits\outputs\preprocessed
Stats dir: D:\fsdf_uca_kits\outputs\stats


## Step 1: Analyze ALL 208 Cases

Compute tumor volume, kidney volume, and tumor/kidney ratio for every case.  
This determines difficulty classification for stratification.

In [2]:
# Analyze every case
all_cases = sorted([d.name for d in PREPROC_DIR.iterdir() if d.is_dir()])
# Exclude cases with known label quality issues
EXCLUDE = {'case_00015', 'case_00037'}
all_cases = [c for c in all_cases if c not in EXCLUDE]
print(f"Total cases: {len(all_cases)} (excluded {len(EXCLUDE)}: {EXCLUDE})")

case_info = []
for i, case_name in enumerate(all_cases):
    label = np.load(PREPROC_DIR / case_name / "label.npy")
    total_voxels = label.size
    kidney_voxels = int((label >= 1).sum())
    tumor_voxels = int((label == 2).sum())
    tumor_ratio = tumor_voxels / max(kidney_voxels, 1)
    
    case_info.append({
        'case_name': case_name,
        'has_tumor': tumor_voxels > 0,
        'tumor_voxels': tumor_voxels,
        'kidney_voxels': kidney_voxels,
        'tumor_ratio': float(tumor_ratio),
        'total_voxels': total_voxels,
    })
    if (i+1) % 50 == 0:
        print(f"  Analyzed {i+1}/{len(all_cases)} cases...")

# Separate tumor vs no-tumor
tumor_cases = [c for c in case_info if c['has_tumor']]
no_tumor_cases = [c for c in case_info if not c['has_tumor']]
print(f"\nTumor cases: {len(tumor_cases)}")
print(f"No-tumor cases: {len(no_tumor_cases)}")

# Tumor volume statistics
tumor_volumes = [c['tumor_voxels'] for c in tumor_cases]
p25 = np.percentile(tumor_volumes, 25)
p50 = np.percentile(tumor_volumes, 50)
p75 = np.percentile(tumor_volumes, 75)

print(f"\nTumor volume distribution (all {len(tumor_cases)} tumor cases):")
print(f"  Min:    {min(tumor_volumes):,}")
print(f"  P25:    {p25:,.0f}")
print(f"  Median: {p50:,.0f}")
print(f"  P75:    {p75:,.0f}")
print(f"  Max:    {max(tumor_volumes):,}")

Total cases: 208 (excluded 2: {'case_00037', 'case_00015'})
  Analyzed 50/208 cases...
  Analyzed 100/208 cases...
  Analyzed 150/208 cases...
  Analyzed 200/208 cases...

Tumor cases: 208
No-tumor cases: 0

Tumor volume distribution (all 208 tumor cases):
  Min:    342
  P25:    4,313
  Median: 16,560
  P75:    50,112
  Max:    807,488


## Step 2: Classify Cases by Difficulty

Using tumor volume quartiles:
- **Hard:** Bottom 25% (smallest tumors — hardest to segment)
- **Medium:** Middle 50%
- **Easy:** Top 25% (largest tumors — easiest to segment)

Validated against known Dice scores from NB04p evaluation.

In [3]:
# Classify by COMBINED difficulty: tumor volume + tumor/kidney ratio
# Hard = very small tumor OR extreme tumor ratio
# This captures both tiny-tumor failures AND morphological outliers

tumor_volumes = [c['tumor_voxels'] for c in tumor_cases]
tumor_ratios = [c['tumor_ratio'] for c in tumor_cases]

vol_p25 = np.percentile(tumor_volumes, 25)
vol_p75 = np.percentile(tumor_volumes, 75)
ratio_p75 = np.percentile(tumor_ratios, 75)  # high ratio = tumor-dominant kidney

# Combined difficulty rule:
#   Hard:   small tumor (< vol_p25) OR extreme ratio (> ratio_p75 AND small volume)
#   Easy:   large tumor (> vol_p75) AND normal ratio
#   Medium: everything else
hard = []
medium = []
easy = []

for c in tumor_cases:
    is_small = c['tumor_voxels'] < vol_p25
    is_large = c['tumor_voxels'] >= vol_p75
    is_extreme_ratio = c['tumor_ratio'] > ratio_p75
    
    if is_small or (is_extreme_ratio and c['tumor_voxels'] < vol_p75):
        hard.append(c)
    elif is_large and not is_extreme_ratio:
        easy.append(c)
    else:
        medium.append(c)

print(f"Difficulty classification (tumor volume + ratio):")
print(f"  Volume P25: {vol_p25:,.0f}, P75: {vol_p75:,.0f}")
print(f"  Ratio P75:  {ratio_p75:.4f}")
print(f"  Hard:   {len(hard)} cases (small tumor or extreme ratio)")
print(f"  Medium: {len(medium)} cases")
print(f"  Easy:   {len(easy)} cases (large tumor, normal ratio)")

# Validate against known Dice scores
print(f"\n--- Validation against known NB04p Dice scores ---")
try:
    tta_path = OUTPUT_DIR / 'results' / 'fold0_nb04p_tta_pp.json'
    if tta_path.exists():
        with open(tta_path) as f:
            data = json.load(f)
        known_scores = {r['case_name']: r['tumor_dice'] for r in data['tta_pp']}
    else:
        e2e_path = OUTPUT_DIR / 'results' / 'fold0_nb04p_e2e.json'
        if e2e_path.exists():
            with open(e2e_path) as f:
                results = json.load(f)
            known_scores = {r['case_name']: r['tumor_dice'] for r in results}
        else:
            known_scores = {}
    
    for label, group in [('Hard', hard), ('Medium', medium), ('Easy', easy)]:
        scores = [known_scores[c['case_name']] for c in group if c['case_name'] in known_scores]
        n_known = len(scores)
        n_unknown = len(group) - n_known
        if scores:
            print(f"  {label:>8}: avg Dice={np.mean(scores):.4f} (n={n_known} known, {n_unknown} unknown)")
        else:
            print(f"  {label:>8}: all {len(group)} cases were in old training set (no scores)")
    
    print(f"\n  Correlation check: Hard avg << Easy avg confirms valid stratification.")
except Exception as e:
    print(f"  Could not load previous results: {e}")
    known_scores = {}

# Show hardest cases
print(f"\nHardest 10 cases (smallest tumors + extreme ratios):")
hard_sorted = sorted(hard, key=lambda x: x['tumor_voxels'])
for c in hard_sorted[:10]:
    score_str = f"Dice={known_scores.get(c['case_name'], 'unknown')}"
    print(f"  {c['case_name']}: vol={c['tumor_voxels']:>6,}, ratio={c['tumor_ratio']:.4f}, {score_str}")

Difficulty classification (tumor volume + ratio):
  Volume P25: 4,313, P75: 50,112
  Ratio P75:  0.1769
  Hard:   56 cases (small tumor or extreme ratio)
  Medium: 148 cases
  Easy:   4 cases (large tumor, normal ratio)

--- Validation against known NB04p Dice scores ---
      Hard: avg Dice=0.4720 (n=19 known, 37 unknown)
    Medium: avg Dice=0.7974 (n=50 known, 98 unknown)
      Easy: avg Dice=0.7771 (n=1 known, 3 unknown)

  Correlation check: Hard avg << Easy avg confirms valid stratification.

Hardest 10 cases (smallest tumors + extreme ratios):
  case_00120: vol=   342, ratio=0.0025, Dice=unknown
  case_00165: vol=   416, ratio=0.0022, Dice=unknown
  case_00072: vol=   919, ratio=0.0041, Dice=unknown
  case_00023: vol=   952, ratio=0.0032, Dice=unknown
  case_00019: vol= 1,105, ratio=0.0038, Dice=0.6290322580645161
  case_00123: vol= 1,138, ratio=0.0057, Dice=unknown
  case_00137: vol= 1,143, ratio=0.0062, Dice=0.0
  case_00209: vol= 1,229, ratio=0.0047, Dice=unknown
  case_00121

## Step 3: Create Stratified 80/20 Split

Proportional sampling from each difficulty bucket:
- Hard: 80% train, 20% val
- Medium: 80% train, 20% val
- Easy: 80% train, 20% val
- No-tumor: 80% train, 20% val

This ensures val set has proportional difficulty distribution.  
**Seed = 42 for reproducibility.**

In [4]:
np.random.seed(42)  # REPRODUCIBLE

def split_80_20(case_list):
    """Split list into 80% train, 20% val. At least 1 in each."""
    n = len(case_list)
    n_val = min(max(1, round(n * 0.20)), n - 1)  # at least 1 in train AND val
    idx = np.random.permutation(n)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    return [case_list[i] for i in train_idx], [case_list[i] for i in val_idx]

# Split each difficulty bucket proportionally
hard_train, hard_val = split_80_20(hard)
medium_train, medium_val = split_80_20(medium)
easy_train, easy_val = split_80_20(easy)
notumor_train, notumor_val = split_80_20(no_tumor_cases)

# Combine into final train/val lists
train_cases = sorted(
    [c['case_name'] for c in hard_train + medium_train + easy_train] +
    [c['case_name'] for c in notumor_train]
)
val_cases = sorted(
    [c['case_name'] for c in hard_val + medium_val + easy_val] +
    [c['case_name'] for c in notumor_val]
)

print(f"{'='*60}")
print(f"STRATIFIED 80/20 SPLIT (seed=42)")
print(f"{'='*60}")
print(f"  Train: {len(train_cases)} cases")
print(f"  Val:   {len(val_cases)} cases")
print(f"  Ratio: {len(train_cases)/len(train_cases+val_cases)*100:.1f}% / {len(val_cases)/len(train_cases+val_cases)*100:.1f}%")

print(f"\n  Val set composition:")
print(f"    Hard tumor:     {len(hard_val)} cases ({len(hard_val)}/{len(hard)} = {100*len(hard_val)/max(len(hard),1):.0f}%)")
print(f"    Medium tumor:   {len(medium_val)} cases ({len(medium_val)}/{len(medium)} = {100*len(medium_val)/max(len(medium),1):.0f}%)")
print(f"    Easy tumor:     {len(easy_val)} cases ({len(easy_val)}/{len(easy)} = {100*len(easy_val)/max(len(easy),1):.0f}%)")
print(f"    No-tumor:       {len(notumor_val)} cases")
print(f"    Total val tumor cases: {len(hard_val)+len(medium_val)+len(easy_val)}")

# Compare with old split
print(f"\n  OLD SPLIT: 138 train / 70 val")
print(f"  NEW SPLIT: {len(train_cases)} train / {len(val_cases)} val")
print(f"  Extra training cases: +{len(train_cases) - 138}")

STRATIFIED 80/20 SPLIT (seed=42)
  Train: 166 cases
  Val:   42 cases
  Ratio: 79.8% / 20.2%

  Val set composition:
    Hard tumor:     11 cases (11/56 = 20%)
    Medium tumor:   30 cases (30/148 = 20%)
    Easy tumor:     1 cases (1/4 = 25%)
    No-tumor:       0 cases
    Total val tumor cases: 42

  OLD SPLIT: 138 train / 70 val
  NEW SPLIT: 166 train / 42 val
  Extra training cases: +28


## Step 4: Val Set Detailed Analysis

Inspect every val case — especially hard ones.  
Check which hard cases ended up in val and which moved to train.

In [5]:
# Detailed val set inspection
print(f"VAL SET — Hard cases (smallest tumors, most likely to fail):")
print(f"{'Case':<15} {'Tumor voxels':>14} {'Tumor ratio':>13} {'Known Dice':>12}")
print("-" * 58)
for c in hard_val:
    score = known_scores.get(c['case_name'], None)
    score_str = f"{score:.4f}" if score is not None else "unknown"
    print(f"  {c['case_name']:<15} {c['tumor_voxels']:>12,} {c['tumor_ratio']:>12.4f} {score_str:>12}")

print(f"\nTRAIN SET — Hard cases (removed from val):")
for c in hard_train:
    score = known_scores.get(c['case_name'], None)
    score_str = f"{score:.4f}" if score is not None else "unknown"
    print(f"  {c['case_name']:<15} {c['tumor_voxels']:>12,} {c['tumor_ratio']:>12.4f} {score_str:>12}")

# Estimate expected score
print(f"\n{'='*60}")
print(f"EXPECTED PERFORMANCE (same model, new val split)")
print(f"{'='*60}")
val_tumor_cases_all = hard_val + medium_val + easy_val
val_known = [(c['case_name'], known_scores[c['case_name']]) 
             for c in val_tumor_cases_all if c['case_name'] in known_scores]
val_unknown = [c['case_name'] for c in val_tumor_cases_all if c['case_name'] not in known_scores]

if val_known:
    known_mean = np.mean([s for _, s in val_known])
    print(f"  Known val cases: {len(val_known)} (from old 70-case val)")
    print(f"  Unknown val cases: {len(val_unknown)} (were in old train — difficulty unknown)")
    print(f"  Mean of known cases: {known_mean:.4f}")
    print(f"  Known val subset mean:            {known_mean:.4f}")
    print(f"\n  NOTE: Unknown cases could be hard or easy.")
    print(f"  Actual improvement will be measured after retraining.")
else:
    print(f"  No known scores available for estimation.")

# List unknown val cases (need attention)
if val_unknown:
    print(f"\n  Unknown val cases (were in old training set):")
    for name in val_unknown:
        info = [c for c in val_tumor_cases_all if c['case_name'] == name][0]
        print(f"    {name}: tumor={info['tumor_voxels']:,} voxels — difficulty UNKNOWN")

VAL SET — Hard cases (smallest tumors, most likely to fail):
Case              Tumor voxels   Tumor ratio   Known Dice
----------------------------------------------------------
  case_00000             4,307       0.0201      unknown
  case_00020             1,826       0.0081       0.2189
  case_00125             3,897       0.0142      unknown
  case_00045            35,632       0.2420      unknown
  case_00075             2,807       0.0155       0.6116
  case_00198             2,535       0.0109      unknown
  case_00133            34,880       0.1829      unknown
  case_00101             2,768       0.0134      unknown
  case_00164             2,549       0.0119      unknown
  case_00040             3,278       0.0165       0.8781
  case_00208             1,916       0.0088      unknown

TRAIN SET — Hard cases (removed from val):
  case_00017             3,502       0.0153       0.6674
  case_00129             3,506       0.0155       0.0000
  case_00121             1,258       

## Step 5: Save Split

Saves to `fold_splits_80_20.json` — separate from old `fold_splits.json`.  
Old split is preserved unchanged.

In [6]:
# Save the split
split_data = {
    'description': 'Difficulty-stratified 80/20 split based on tumor volume and tumor/kidney ratio',
    'seed': 42,
    'split_criteria': 'tumor_voxels + tumor_ratio combined',
    'quartiles': {
        'vol_p25': float(vol_p25), 
        'vol_p75': float(vol_p75),
        'ratio_p75': float(ratio_p75)
    },
    'folds': {
        'fold_0': {
            'train': train_cases,
            'val': val_cases
        }
    },
    'val_composition': {
        'hard_tumor': [c['case_name'] for c in hard_val],
        'medium_tumor': [c['case_name'] for c in medium_val],
        'easy_tumor': [c['case_name'] for c in easy_val],
        'no_tumor': [c['case_name'] for c in notumor_val],
    },
    'case_info': {c['case_name']: {
        'tumor_voxels': c['tumor_voxels'],
        'kidney_voxels': c['kidney_voxels'],
        'tumor_ratio': c['tumor_ratio'],
        'difficulty': (
            'hard' if c['case_name'] in {x['case_name'] for x in hard}
            else 'easy' if c['case_name'] in {x['case_name'] for x in easy}
            else 'medium'
        )
    } for c in tumor_cases}
}

save_path = STATS_DIR / 'fold_splits_80_20.json'
with open(save_path, 'w') as f:
    json.dump(split_data, f, indent=2)
print(f"Saved: {save_path}")

# Verification
overlap = set(train_cases) & set(val_cases)
union = set(train_cases) | set(val_cases)
all_set = set([c['case_name'] for c in case_info])

print(f"\nVERIFICATION:")
print(f"  Train cases: {len(train_cases)}")
print(f"  Val cases:   {len(val_cases)}")
print(f"  Overlap:     {len(overlap)} (MUST be 0)")
print(f"  Union:       {len(union)} (MUST be {len(all_set)})")
print(f"  Missing:     {len(all_set - union)} (MUST be 0)")

if len(overlap) == 0 and len(union) == len(all_set):
    print(f"\n  ✓ Split is VALID")
else:
    print(f"\n  ✗ Split has ERRORS — check above!")

# Summary comparison
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"  Old split: 138 train / 70 val (fold_splits.json)")
print(f"  New split: {len(train_cases)} train / {len(val_cases)} val (fold_splits_80_20.json)")
print(f"  Shared: Both use ALL {len(all_set)} preprocessed cases")
print(f"  Old split file: UNCHANGED")
print(f"\n  Next step: NB07b — Retrain nnU-Net Stage 1 on {len(train_cases)} cases")

Saved: D:\fsdf_uca_kits\outputs\stats\fold_splits_80_20.json

VERIFICATION:
  Train cases: 166
  Val cases:   42
  Overlap:     0 (MUST be 0)
  Union:       208 (MUST be 208)
  Missing:     0 (MUST be 0)

  ✓ Split is VALID

SUMMARY
  Old split: 138 train / 70 val (fold_splits.json)
  New split: 166 train / 42 val (fold_splits_80_20.json)
  Shared: Both use ALL 208 preprocessed cases
  Old split file: UNCHANGED

  Next step: NB07b — Retrain nnU-Net Stage 1 on 166 cases
